In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from collections import Counter

class SimpleBaggingClassifier:
    def __init__(self, n_estimators=20, max_depth=None, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        X, y = np.asarray(X), np.asarray(y)
        n = len(y)
        self.trees = []

        for _ in range(self.n_estimators):
            # Bootstrap: n draws with replacement
            idx = rng.integers(0, n, size=n)

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth, random_state=int(rng.integers(0, 2**31 - 1)))
            tree.fit(X[idx], y[idx])
            self.trees.append(tree)

        return self

    def predict(self, X):
        X = np.asarray(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])

        # Majority vote for each example
        return np.array([
            Counter(predictions[:, i]).most_common(1)[0][0]
            for i in range(X.shape[0])
        ])